# Section 1.3 — Processing times under the active event lifecycle

This report-facing notebook summarizes the evidence introduced on `refac/event_lifecycle`. Processing time is no longer the wall-clock span from the first start to a terminal event. For `W_` work items it is the sum of active `start/resume → suspend/complete/ate_abort` sessions; suspended and queueing time are separate lifecycle components.

All figures use the TUM palette sourced from the companion report repository and are exported to `visualization/` as both PDF and SVG. The notebook reads versioned validation artifacts only, so it does not retrain a model or rerun the simulator.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'analysis').is_dir() and (path / 'simulation').is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.tum_style import (
    TUM_BLUE, TUM_BLUE_LIGHT, TUM_GRAY, TUM_GREEN, TUM_ORANGE,
    TUM_RED, TUM_TEAL, TUM_VIOLET, apply_tum_style, save_figure,
)

apply_tum_style()
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

VALIDATION_DIR = ROOT / 'output/validation/lifecycle_active'
MODEL_METRICS_PATH = ROOT / 'output/models/processing_time_metrics_active.json'
REPORT_INPUT_DIR = ROOT / 'output/report_inputs'
REPORT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
MODE_FILES = {
    'Distribution': VALIDATION_DIR / 'distribution.json',
    'Point ML': VALIDATION_DIR / 'ml_model.json',
    'Probabilistic ML': VALIDATION_DIR / 'ml_probabilistic.json',
}
missing = [path for path in (*MODE_FILES.values(), MODEL_METRICS_PATH) if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing lifecycle validation artifacts: {missing}')

runs = {label: json.loads(path.read_text()) for label, path in MODE_FILES.items()}
model_metrics = json.loads(MODEL_METRICS_PATH.read_text())
REFERENCE_RESUME_RATE = 0.175  # mined BPIC-17 baseline documented by the lifecycle branch
print(f'Loaded {len(runs)} active-lifecycle validation runs from {VALIDATION_DIR.relative_to(ROOT)}')

## 1. The lifecycle changes the estimand

The refactor makes three clocks explicit:

1. **Queueing:** `schedule → start` or readiness after a suspension until reacquisition.
2. **Active service:** each `start/resume → suspend/terminal` interval.
3. **External/non-active waiting:** the remaining time while a work item is suspended.

Only active service is fitted by the processing-time models. Lifecycle hazards separately determine suspension, resumption, abort, completion, and withdrawal. This prevents nights and customer waits from being mislabelled as resource-occupied processing time.

Coverage depends on the lifecycle mode. The active artifact covers all eight W_ activities. The standalone CLI nevertheless defaults to legacy mode, whereas the policy evaluation notebooks explicitly select active mode. A_/O_ state changes have no observed start/complete pairs and therefore retain assumed fallback durations; the evaluation reports W_-only KPIs and a zero-duration A_/O_ sensitivity alongside the main specification.

In [ ]:
from simulation.components.process import (
    FALLBACK_MEAN_DURATIONS, PROCESSING_TIME_PARAMS,
)

active_inputs = json.loads((ROOT / "simulation_inputs_active.json").read_text())["lifecycle"]
active_work = set(active_inputs["processing_times"])
coverage_audit = pd.Series({
    "active fitted W_ durations": len(active_work),
    "active W_ session-end hazards": len(active_inputs["session_end_probs"]),
    "active W_ suspend-end hazards": len(active_inputs["suspend_end_probs"]),
    "active W_ resume-gap fits": len(active_inputs["resume_gap_params"]),
    "active W_ withdrawal hazards": len(active_inputs["withdraw_hazard"]),
    "legacy fitted W_ durations": sum(a.startswith("W_") for a in PROCESSING_TIME_PARAMS),
    "fallback W_ durations": sum(a.startswith("W_") for a in FALLBACK_MEAN_DURATIONS),
    "fallback A_/O_ durations": sum(a.startswith(("A_", "O_")) for a in FALLBACK_MEAN_DURATIONS),
}, name="activities")
assert len(active_work) == 8
assert all(active_work == set(active_inputs[key]) for key in (
    "session_end_probs", "suspend_end_probs", "resume_gap_params"
))
coverage_audit.to_csv(REPORT_INPUT_DIR / 'processing_time_coverage.csv')
display(coverage_audit.to_frame())

boundary_hazards = pd.DataFrame({
    "P(complete | session end)": active_inputs["session_end_probs"],
    "P(resume | suspended end)": active_inputs["suspend_end_probs"],
}).query("`P(complete | session end)` in [0, 1] or `P(resume | suspended end)` in [0, 1]")
boundary_hazards.to_csv(REPORT_INPUT_DIR / 'processing_time_boundary_hazards.csv')
display(boundary_hazards)

In [ ]:
summary_rows = []
for mode, run in runs.items():
    general = run['general_metrics']
    lifecycle = run['lifecycle']
    summary_rows.append({
        'mode': mode,
        'completed cases': run['configuration']['completed_cases'],
        'processing-time mean relative error': general['processing_times']['mean_rel_err'],
        'suspend-count MAE': general['lifecycle']['suspend_counts']['mean_abs_error'],
        'same-resource resume rate': lifecycle['resume_ownership']['same_resource_rate'],
        'case-duration relative error': general['case_stats']['case_duration_rel_err'],
        'branching mean TVD': general['branching']['mean_tvd'],
    })
summary = pd.DataFrame(summary_rows).set_index('mode')
summary.to_csv(REPORT_INPUT_DIR / 'processing_time_sampler_summary.csv')
display(summary.round(3))

In [ ]:
plot_metrics = summary[[
    'processing-time mean relative error', 'suspend-count MAE'
]].rename(columns={
    'processing-time mean relative error': 'active-time mean relative error',
    'suspend-count MAE': 'mean absolute suspend-count error',
})

fig, ax = plt.subplots(figsize=(8.6, 4.2))
x = np.arange(len(plot_metrics))
width = 0.34
ax.bar(x - width/2, plot_metrics.iloc[:, 0], width, color=TUM_BLUE, label=plot_metrics.columns[0])
ax.bar(x + width/2, plot_metrics.iloc[:, 1], width, color=TUM_ORANGE, label=plot_metrics.columns[1])
ax.set(xticks=x, xticklabels=plot_metrics.index, ylabel='error (lower is better)',
       title='Active-time fit and lifecycle churn must be evaluated separately')
ax.legend()
plt.tight_layout()
save_figure(fig, '03_active_time_and_churn_error')
plt.show()

**Finding.** The activity-level distribution baseline has the smallest mean error in active-session means (**21.3%**). Probabilistic ML improves substantially over point ML (**36.8%** versus **74.0%**) because it restores spread that a conditional mean suppresses. Suspend-count calibration is similar across modes (**0.221–0.282 MAE**) because churn is governed by the shared lifecycle mechanism, not the duration sampler.

In [ ]:
activities = sorted({
    activity
    for run in runs.values()
    for activity, values in run['general_metrics']['processing_times']['per_activity'].items()
    if values is not None
})
activity_error = pd.DataFrame({
    mode: {
        activity: run['general_metrics']['processing_times']['per_activity'].get(activity, {}).get('rel_err_mean', np.nan)
        if run['general_metrics']['processing_times']['per_activity'].get(activity) is not None else np.nan
        for activity in activities
    }
    for mode, run in runs.items()
}).loc[activities]

fig, ax = plt.subplots(figsize=(9.4, 4.8))
activity_error.plot.bar(ax=ax, color=[TUM_BLUE, TUM_ORANGE, TUM_TEAL], width=0.78)
ax.set(ylabel='relative error of mean active-session time', xlabel='',
       title='Processing-time accuracy differs by activity and sampler')
ax.set_xticklabels([name.removeprefix('W_') for name in activity_error.index], rotation=20, ha='right')
ax.legend(title='duration sampler', loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
save_figure(fig, '03_processing_time_error_by_activity')
plt.show()

## 2. Active time is only a small part of elapsed lifecycle time

The terminal recomposition below is the central lifecycle result. It sums active sessions and compares them with the non-active remainder of the same work items. Means are used here because the report needs to expose the operational burden of the long waiting tail; medians remain much smaller and are available in the validation JSON.

In [ ]:
recomposition_rows = []
for mode, run in runs.items():
    for terminal in ('complete', 'ate_abort'):
        values = run['lifecycle']['terminal_recomposition'][terminal]
        recomposition_rows.append({
            'mode': mode,
            'terminal': terminal.replace('_', ' '),
            'active hours': values['active_seconds']['mean'] / 3600,
            'non-active hours': values['non_active_seconds']['mean'] / 3600,
        })
recomposition = pd.DataFrame(recomposition_rows)
recomposition['label'] = recomposition['mode'] + ', ' + recomposition['terminal']
recomposition.to_csv(REPORT_INPUT_DIR / 'processing_time_recomposition.csv', index=False)

fig, ax = plt.subplots(figsize=(3.35, 3.0))
y = np.arange(len(recomposition))[::-1]
ax.barh(y, recomposition['active hours'], color=TUM_BLUE, label='active service')
ax.barh(y, recomposition['non-active hours'], left=recomposition['active hours'],
        color=TUM_BLUE_LIGHT, label='non-active waiting')
ax.set(yticks=y, yticklabels=recomposition['label'], xlabel='mean hours per work item', ylabel='')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8,
          frameon=False, columnspacing=0.8, handlelength=1.4)
ax.tick_params(labelsize=8.5)
plt.tight_layout()
save_figure(fig, '03_lifecycle_time_recomposition')
plt.show()

**Finding.** For completed items, mean active service ranges from roughly **2.3 to 13.1 minutes**, while mean non-active time remains about **7.2 to 9.0 hours**. The separation is therefore not cosmetic: using elapsed time as processing time would allocate most of a work item's lifetime to a resource even though the item is suspended or waiting.

In [ ]:
resume_rates = summary['same-resource resume rate']
fig, ax = plt.subplots(figsize=(8.4, 4.0))
bars = ax.bar(resume_rates.index, resume_rates.values * 100,
              color=[TUM_BLUE, TUM_ORANGE, TUM_TEAL])
ax.axhline(REFERENCE_RESUME_RATE * 100, color=TUM_RED, linestyle='--', linewidth=1.5,
           label=f'BPIC-17 reference ({REFERENCE_RESUME_RATE:.1%})')
ax.bar_label(bars, labels=[f'{value:.1%}' for value in resume_rates], padding=3)
ax.set(ylabel='resumes retaining the same resource (%)', ylim=(0, 22),
       title='Resumed work usually returns to the pool, but continuity is understated')
ax.legend()
plt.tight_layout()
save_figure(fig, '03_resume_resource_continuity')
plt.show()

**Finding.** Only **11.1–13.9%** of simulated resumes retain the previous resource, below the mined **17.5%** reference. Pool-based reacquisition is directionally supported—most resumes really do change owner—but the model still understates continuity by **3.6–6.4 percentage points**. This is a calibration gap, not evidence to force same-resource resumption.

## 3. What the fitted ML diagnostics say

In [ ]:
point = model_metrics['point_model']
quantile = model_metrics['quantile_eval']
importance = pd.Series(model_metrics['feature_importances']).rename(index={
    'previous_activity_enc': 'previous activity',
    'resource_enc': 'resource',
    'case_age_seconds': 'case age',
    'n_previous_activities': 'previous activities',
    'case_position': 'case position',
    'hour_of_day': 'hour of day',
    'activity_enc': 'activity',
    'day_of_week': 'day of week',
}).sort_values()

fig, ax = plt.subplots(figsize=(3.35, 2.8))
importance.plot.barh(ax=ax, color=TUM_BLUE)
ax.set(xlabel='relative importance', ylabel='')
ax.tick_params(labelsize=8.5)
plt.tight_layout()
save_figure(fig, '03_point_model_feature_importance')
plt.show()

coverage = pd.Series({
    '50% interval': quantile['coverage_50pct_interval'],
    '90% interval': quantile['coverage_90pct_interval'],
})
targets = np.array([0.50, 0.90])
x = np.arange(len(coverage))
fig, ax = plt.subplots(figsize=(3.35, 2.45))
ax.bar(x - 0.18, coverage.values * 100, 0.36, color=TUM_TEAL, label='observed')
ax.bar(x + 0.18, targets * 100, 0.36, color=TUM_GRAY, label='nominal')
ax.set(xticks=x, xticklabels=coverage.index, ylabel='coverage (%)', ylim=(0, 100))
ax.legend(loc='upper left', fontsize=8)
ax.tick_params(labelsize=8.5)
plt.tight_layout()
save_figure(fig, '03_quantile_interval_coverage')
plt.show()

model_diagnostics = pd.DataFrame([{
    'temporal split': point['split'],
    'test sessions': point['n_test'],
    'log R²': point['r2_log'],
    'raw R²': point['r2_raw'],
    'MAE (minutes)': point['mae_seconds'] / 60,
    '90% coverage': quantile['coverage_90pct_interval'],
    '50% coverage': quantile['coverage_50pct_interval'],
}])
model_diagnostics.to_csv(REPORT_INPUT_DIR / 'processing_time_ml_diagnostics.csv', index=False)
display(model_diagnostics)

**Finding.** Point prediction has little temporal explanatory power (`R²_log = 0.087`, negative raw-scale `R²`) and should not be presented as a strong duration predictor. The probabilistic model is more defensible for simulation: its central intervals cover **49.6%** and **88.0%** against nominal 50% and 90% targets. Previous activity, resource, and case age account for most point-model importance, but importance is descriptive rather than causal.

## 4. Report conclusion and limitations

- **Use the distribution sampler as the basic model.** It best reproduces mean active-session duration in these runs and retains realistic variability.
- **Use probabilistic ML as the advanced model.** It is less accurate on activity means than the distribution baseline, but substantially better than point ML and its prediction intervals are calibrated.
- **Distinguish validation from execution.** The policy evaluation notebooks explicitly use the distribution sampler. The ML results in this notebook support a model comparison; they do not show that the reported policy experiments use ML durations.
- **Keep lifecycle and duration evaluation separate.** Similar suspend-count errors across samplers show that duration fit does not validate churn.
- **Separate fitted work from assumed state-change time.** All eight active W_ activities have fitted session parameters, whereas A_/O_ durations are assumptions because the log has no start/complete pairs for them. Report W_-only resource KPIs and the paired zero-duration A_/O_ sensitivity with the main all-activity results.
- **Treat boundary hazards as sparse-data warnings.** `W_Shortened completion ` and `W_Personal Loan collection` currently have `P(complete)=0` and `P(resume)=1`. Evaluation records their route counts and rejects runs in which the 60-session guard forces completion; the probabilities should only be changed through a documented re-estimation or smoothing rule.
- **Do not claim the overall process is calibrated.** Mean case-duration relative error remains about **90%**, ten reference activities are absent from each run, and `W_Complete application` suspend counts are materially underestimated. Those are process/lifecycle limitations beyond the active-session sampler.

The figures generated by this notebook are report-ready inputs, but the limitations above should accompany them in the report text. The final cell writes a compact machine-readable hand-off so report values do not have to be copied from rendered notebook tables.

In [ ]:
completed_recomposition = recomposition[recomposition['terminal'] == 'complete']
report_values = {
    'selected_basic_sampler': summary['processing-time mean relative error'].idxmin(),
    'active_time_mean_relative_error': {
        mode: float(value)
        for mode, value in summary['processing-time mean relative error'].items()
    },
    'completed_active_minutes_range': [
        float(completed_recomposition['active hours'].min() * 60),
        float(completed_recomposition['active hours'].max() * 60),
    ],
    'completed_non_active_hours_range': [
        float(completed_recomposition['non-active hours'].min()),
        float(completed_recomposition['non-active hours'].max()),
    ],
    'historical_same_resource_resume_rate': REFERENCE_RESUME_RATE,
    'simulated_same_resource_resume_rate_range': [
        float(summary['same-resource resume rate'].min()),
        float(summary['same-resource resume rate'].max()),
    ],
    'boundary_hazard_activities': boundary_hazards.index.tolist(),
    'policy_evaluation_processing_time_mode': 'distribution',
    'report_figures': [
        '03_lifecycle_time_recomposition.pdf',
        '03_point_model_feature_importance.pdf',
        '03_quantile_interval_coverage.pdf',
    ],
}
(REPORT_INPUT_DIR / 'processing_time_report_values.json').write_text(
    json.dumps(report_values, indent=2) + '\n'
)
display(pd.Series(report_values, name='report value'))